# Olist E-commerce: Customer Analytics

**Questions:** Do customers come back? How long until a second order?
How does retention look by cohort? Who are the best, at-risk and lost customers (RFM)?

**Customer = `customer_unique_id`** (the real person; `customer_id` is created per order).

In [1]:
import duckdb

%load_ext sql
con = duckdb.connect("../data/olist.duckdb", read_only=True)
%sql con
%config SqlMagic.displaylimit = 50

In [2]:
%%sql
WITH per_customer_id AS (
    SELECT customer_id, COUNT(*) AS orders
    FROM orders
    WHERE order_status = 'delivered'
      AND order_purchase_timestamp >= '2017-01-01'
      AND order_purchase_timestamp <  '2018-09-01'
    GROUP BY customer_id
)
SELECT COUNT(*) AS customers,
       SUM(CASE WHEN orders > 1 THEN 1 ELSE 0 END) AS repeat_customers
FROM per_customer_id;

Running query in 'DuckDBPyConnection'

customers,repeat_customers
96211,0


In [3]:
%%sql
WITH customer_orders AS (
    SELECT c.customer_unique_id,
           o.order_id,
           o.order_purchase_timestamp AS order_date,
           SUM(i.price) AS order_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp
),
per_customer AS (
    SELECT customer_unique_id,
           COUNT(*) AS orders,
           SUM(order_value) AS spend
    FROM customer_orders
    GROUP BY customer_unique_id
)
SELECT COUNT(*) AS customers,
       SUM(CASE WHEN orders > 1 THEN 1 ELSE 0 END) AS repeat_customers,
       ROUND(100.0 * SUM(CASE WHEN orders > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS repeat_pct,
       ROUND(100.0 * SUM(CASE WHEN orders > 1 THEN spend ELSE 0 END) / SUM(spend), 1) AS repeat_revenue_pct,
       MAX(orders) AS max_orders
FROM per_customer;

Running query in 'DuckDBPyConnection'

customers,repeat_customers,repeat_pct,repeat_revenue_pct,max_orders
93104,2789,3.0,5.5,15


In [4]:
%%sql
WITH customer_orders AS (
    SELECT c.customer_unique_id,
           o.order_id,
           o.order_purchase_timestamp AS order_date,
           SUM(i.price) AS order_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp
),
numbered AS (
    SELECT customer_unique_id,
           order_date,
           ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY order_date) AS order_number,
           LAG(order_date) OVER (PARTITION BY customer_unique_id ORDER BY order_date) AS previous_order_date
    FROM customer_orders
)
SELECT COUNT(*) AS second_orders,
       MEDIAN(date_diff('day', previous_order_date, order_date)) AS median_days,
       ROUND(AVG(date_diff('day', previous_order_date, order_date)), 0) AS avg_days,
       SUM(CASE WHEN date_diff('day', previous_order_date, order_date) = 0 THEN 1 ELSE 0 END) AS same_day
FROM numbered
WHERE order_number = 2;

Running query in 'DuckDBPyConnection'

second_orders,median_days,avg_days,same_day
2789,29.0,80.0,827


In [5]:
%%sql
WITH customer_orders AS (
    SELECT c.customer_unique_id,
           o.order_id,
           o.order_purchase_timestamp AS order_date
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
),
first_purchase AS (
    SELECT customer_unique_id,
           MIN(date_trunc('month', order_date))::DATE AS cohort_month
    FROM customer_orders
    GROUP BY customer_unique_id
),
activity AS (
    SELECT DISTINCT f.customer_unique_id,
           f.cohort_month,
           date_diff('month', f.cohort_month, date_trunc('month', co.order_date)::DATE) AS month_number
    FROM customer_orders co
    JOIN first_purchase f ON co.customer_unique_id = f.customer_unique_id
),
cohort_size AS (
    SELECT cohort_month, COUNT(*) AS customers
    FROM first_purchase
    GROUP BY cohort_month
)
SELECT a.cohort_month,
       s.customers AS cohort_size,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.month_number = 1 THEN a.customer_unique_id END) / s.customers, 2) AS month_1_pct,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.month_number = 2 THEN a.customer_unique_id END) / s.customers, 2) AS month_2_pct,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.month_number = 3 THEN a.customer_unique_id END) / s.customers, 2) AS month_3_pct,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.month_number BETWEEN 1 AND 6 THEN a.customer_unique_id END) / s.customers, 2) AS within_6_months_pct
FROM activity a
JOIN cohort_size s ON a.cohort_month = s.cohort_month
GROUP BY a.cohort_month, s.customers
ORDER BY a.cohort_month;

Running query in 'DuckDBPyConnection'

cohort_month,cohort_size,month_1_pct,month_2_pct,month_3_pct,within_6_months_pct
2017-01-01,718,0.28,0.28,0.14,1.67
2017-02-01,1628,0.18,0.31,0.12,1.41
2017-03-01,2503,0.44,0.36,0.4,1.76
2017-04-01,2257,0.62,0.22,0.18,1.86
2017-05-01,3451,0.46,0.46,0.29,2.0
2017-06-01,3037,0.49,0.4,0.43,2.07
2017-07-01,3753,0.53,0.35,0.24,1.87
2017-08-01,4057,0.69,0.35,0.27,2.29
2017-09-01,4005,0.7,0.55,0.27,2.17
2017-10-01,4328,0.72,0.25,0.09,1.64


In [6]:
%%sql
WITH customer_orders AS (
    SELECT c.customer_unique_id,
           o.order_id,
           o.order_purchase_timestamp AS order_date,
           SUM(i.price) AS order_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp
),
rfm AS (
    SELECT customer_unique_id,
           date_diff('day', MAX(order_date), DATE '2018-09-01') AS recency_days,
           COUNT(*) AS frequency,
           ROUND(SUM(order_value), 2) AS monetary
    FROM customer_orders
    GROUP BY customer_unique_id
),
scored AS (
    SELECT *,
           NTILE(5) OVER (ORDER BY recency_days DESC, customer_unique_id) AS r_score,
           NTILE(5) OVER (ORDER BY monetary, customer_unique_id) AS m_score
    FROM rfm
)
SELECT m_score,
       COUNT(*) AS customers,
       MIN(monetary) AS min_spend,
       MAX(monetary) AS max_spend
FROM scored
GROUP BY m_score
ORDER BY m_score;

Running query in 'DuckDBPyConnection'

m_score,customers,min_spend,max_spend
1,18621,0.85,39.9
2,18621,39.9,69.9
3,18621,69.9,109.9
4,18621,109.9,179.9
5,18620,179.9,13440.0


In [7]:
%%sql
WITH customer_orders AS (
    SELECT c.customer_unique_id,
           o.order_id,
           o.order_purchase_timestamp AS order_date,
           SUM(i.price) AS order_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp
),
rfm AS (
    SELECT customer_unique_id,
           date_diff('day', MAX(order_date), DATE '2018-09-01') AS recency_days,
           COUNT(*) AS frequency,
           ROUND(SUM(order_value), 2) AS monetary
    FROM customer_orders
    GROUP BY customer_unique_id
),
scored AS (
    SELECT *,
           NTILE(5) OVER (ORDER BY recency_days DESC, customer_unique_id) AS r_score,
           NTILE(5) OVER (ORDER BY monetary, customer_unique_id) AS m_score
    FROM rfm
),
segmented AS (
    SELECT *,
           CASE WHEN frequency >= 2                 THEN 'Repeat customers'
                WHEN r_score >= 4 AND m_score >= 4  THEN 'Recent big spenders'
                WHEN r_score <= 2 AND m_score >= 4  THEN 'Lapsed big spenders'
                WHEN r_score <= 2 AND m_score <= 2  THEN 'Lost low spenders'
                ELSE 'Regular'
           END AS segment
    FROM scored
)
SELECT segment,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_customers,
       ROUND(SUM(monetary), 0) AS revenue,
       ROUND(100.0 * SUM(monetary) / SUM(SUM(monetary)) OVER (), 1) AS pct_revenue,
       ROUND(AVG(monetary), 0) AS avg_spend,
       ROUND(AVG(recency_days), 0) AS avg_days_since_last_order
FROM segmented
GROUP BY segment
ORDER BY revenue DESC;

Running query in 'DuckDBPyConnection'

segment,customers,pct_customers,revenue,pct_revenue,avg_spend,avg_days_since_last_order
Regular,46931,50.4,4191182.0,31.8,89.0,189.0
Recent big spenders,14441,15.5,3856546.0,29.3,267.0,94.0
Lapsed big spenders,13745,14.8,3800304.0,28.8,276.0,394.0
Repeat customers,2789,3.0,725355.0,5.5,260.0,222.0
Lost low spenders,15198,16.3,607639.0,4.6,40.0,396.0


In [8]:
con.close()

## Findings: customers

- **Wrong ID = wrong answer:** counting by `customer_id` shows 0 repeat customers. Using the real
  person (`customer_unique_id`) shows **93,104 customers, of whom only 2,789 (3.0%) bought again**,
  bringing just 5.5% of revenue. **Olist is a one-time-purchase business.**
- **Second orders:** median **29 days** after the first (average 80, pulled up by late returners).
  **30% of "second orders" were placed the same day** (split baskets), so true loyalty is even lower.
- **Cohorts:** under 1% return in any month; ~1.5–2% within 6 months, with no improvement over time.
  The Black Friday cohort (Nov 2017) retained worse than September's: discounts brought one-time deal hunters.
  Recent cohorts show zeros only because the data ends (right-censoring).
- **RFM segments:**
  - **Recent big spenders** (14,441, avg R$ 267): best target for a second-purchase campaign.
  - **Lapsed big spenders** (13,745, avg R$ 276, 13 months inactive, 29% of revenue): win-back campaign.
  - **Lost low spenders** (15,198, avg R$ 40): low priority.

**Recommendation:** growth stalled in 2018 because Olist depends on new customers. Even moving repeat
purchases from 3% to 10% would be a major revenue lever.

## SQL lessons
- Choose the grain first (here: one row per customer).
- `PARTITION BY` customer is essential for LAG/ROW_NUMBER per customer.
- Conditional aggregation (`COUNT(DISTINCT CASE WHEN …)`) pivots rows into columns.
- `NTILE` creates equal-size groups; don't force it on a column with no variation (frequency).
- **Add a unique tie-breaker** to `NTILE`/`ROW_NUMBER` ordering (here `customer_unique_id`). Without it, customers
  with tied values were split arbitrarily and the segment counts changed on every run.